# DOH 골프 3D 회전 — 쉬운 버전 (가입 없음)

**하는 법:** 위에서부터 각 회색칸 왼쪽 **▶** 를 순서대로 누르세요.
**중요:** 먼저 위 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU → 저장**.

(로그인/가입 필요 없음. 3D 관절만 뽑는 MMPose 모델 = SMPL 등록 불필요.)


### 1칸. 설치 (5~10분, 글자 주르륵=정상)
맨 끝에 **`>>> mmpose OK`** 가 찍히면 성공. **`설치 실패`** 가 뜨면 그 위 빨간 줄을 캡처해 보내주세요.


In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda)   # 이 줄 결과도 알려주면 좋아요
!pip install -q -U openmim
!mim install -q mmengine
!mim install -q 'mmcv>=2.0.1'
!mim install -q 'mmdet>=3.1.0'
!mim install -q 'mmpose>=1.3.0'
try:
    import mmpose, mmcv, mmdet
    print('>>> mmpose OK', mmpose.__version__, '| mmcv', mmcv.__version__)
except Exception as e:
    print('설치 실패 — 위 빨간 줄 캡처:', repr(e))


### 2칸. 스윙 영상 올리기
누르면 파일창 → 골프 스윙 mp4 (짧게 자른 게 빠름).


In [ ]:
from google.colab import files
up = files.upload()
VIDEO = list(up.keys())[0]
print('올린 영상:', VIDEO)


### 3칸. 3D 분석 (제일 오래 걸림)


In [ ]:
try:
    from mmpose.apis import MMPoseInferencer
except ModuleNotFoundError:
    raise SystemExit('❌ 1칸(설치)을 먼저 성공시켜야 해요. 1칸 ▶ 다시 눌러 >>> mmpose OK 확인.')
import numpy as np, pickle

inferencer = MMPoseInferencer(pose3d='human3d', device='cuda')
gen = inferencer(VIDEO, pred_out_dir='mmpose_preds', return_vis=False)

frames = []
for r in gen:
    preds = r['predictions'][0]
    if not preds:
        continue
    kp = np.array(preds[0]['keypoints'])   # (관절, 3) 3D
    frames.append(kp)

J = np.array(frames)
pickle.dump({'joints': J}, open('joints.pkl', 'wb'))
print('프레임', J.shape[0], '· shape', J.shape)


### 4칸. 회전 숫자 뽑기
P4(백스윙탑) 프레임 번호는 영상 보고 대략. 모르면 그대로 둬도 그래프는 나옴.


In [ ]:
!wget -q https://raw.githubusercontent.com/tinyalex3628-dotcom/doh-golf-survey/claude/ai-video-analysis-engine-wlr06k/pose3d_poc/wham_golf_rotation.py -O rot.py
!python rot.py joints.pkl --skeleton h36m --check


In [ ]:
!python rot.py joints.pkl --skeleton h36m --p1 0 --p4 0 --p7 0 --png rot.png
from IPython.display import Image; import os
if os.path.exists('rot.png'): display(Image('rot.png'))


### 5칸. 형한테 보낼 것
- **백스윙탑 흉곽 회전 숫자** + **그래프(rot.png)** 캡처
- 빨간 에러 나오면 그 화면 캡처 → 고쳐서 다시 드림
